# SARIMAX — otimização walk-forward

A seleção usa apenas os 80% de desenvolvimento. Em cada origem de validação o modelo é reajustado e prevê somente a próxima hora.

**Esta etapa é pesada e possui checkpoint em CSV.** Se interromper, rode novamente para continuar dos candidatos já concluídos.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Image

here = Path.cwd().resolve()
group = next(
    (p for p in [here, *here.parents] if (p / "config.py").exists() and (p / "lib").is_dir()),
    None,
)
if group is None:
    raise RuntimeError("Não encontrei analyses/grupo2. Abra o notebook dentro do projeto.")
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib, features, eda_stl
from lib import models_v2 as models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print("outputs →", config.OUTPUT_DIR)


outputs → C:\Users\brunacardoso-ieg\OneDrive - Instituto Germinare\Área de Trabalho\Tech\Séries Temporais\trabalho_PLS_grupo2\pls-regration-comparation\analyses\grupo2\outputs


In [2]:
model_path = require_file(
    config.DATA_DIR / "modeling_frame.csv",
    "01_dados/03_feature_engineering.ipynb",
)
model_df = pd.read_csv(model_path, parse_dates=[config.DATETIME_COL])
protocol = models.protocol_summary(model_df)
print(
    f"linhas={len(model_df)} | validação={protocol['n_validation_origins']} origens "
    f"| teste={protocol['n_test_origins']} origens"
)


linhas=28636 | validação=258 origens | teste=521 origens


In [3]:
ranking_path = config.RESULT_DIR / "tuning_candidates_SARIMAX.csv"
screening_path = config.RESULT_DIR / "sarimax_bic_screening.csv"
winner, ranking, screening = models.tune_sarimax(model_df, screening_path, ranking_path)
save_json(winner, config.RESULT_DIR / "best_params_SARIMAX.json")
all_params_path = config.RESULT_DIR / "best_params.json"
all_params = load_json(all_params_path) if all_params_path.exists() else {}
all_params["SARIMAX"] = winner
save_json(all_params, all_params_path)
print("Vencedor:", winner)
display(ranking)


SARIMAX BIC 1/72: {'order': (0, 0, 0), 'seasonal_order': (0, 0, 0, 24)}
SARIMAX BIC 2/72: {'order': (0, 0, 0), 'seasonal_order': (1, 0, 0, 24)}
SARIMAX BIC 3/72: {'order': (0, 0, 0), 'seasonal_order': (0, 0, 1, 24)}
SARIMAX BIC 4/72: {'order': (0, 0, 0), 'seasonal_order': (0, 1, 1, 24)}
SARIMAX BIC 5/72: {'order': (0, 0, 1), 'seasonal_order': (0, 0, 0, 24)}
SARIMAX BIC 6/72: {'order': (0, 0, 1), 'seasonal_order': (1, 0, 0, 24)}
SARIMAX BIC 7/72: {'order': (0, 0, 1), 'seasonal_order': (0, 0, 1, 24)}
SARIMAX BIC 8/72: {'order': (0, 0, 1), 'seasonal_order': (0, 1, 1, 24)}
SARIMAX BIC 9/72: {'order': (0, 0, 2), 'seasonal_order': (0, 0, 0, 24)}
SARIMAX BIC 10/72: {'order': (0, 0, 2), 'seasonal_order': (1, 0, 0, 24)}
SARIMAX BIC 11/72: {'order': (0, 0, 2), 'seasonal_order': (0, 0, 1, 24)}
SARIMAX BIC 12/72: {'order': (0, 0, 2), 'seasonal_order': (0, 1, 1, 24)}
SARIMAX BIC 13/72: {'order': (0, 1, 0), 'seasonal_order': (0, 0, 0, 24)}
SARIMAX BIC 14/72: {'order': (0, 1, 0), 'seasonal_order': (1

C:\Users\brunacardoso-ieg\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


SARIMAX: 40/258 origens | MAE parcial=329.924
SARIMAX: 60/258 origens | MAE parcial=330.817
SARIMAX: 80/258 origens | MAE parcial=325.510
SARIMAX: 100/258 origens | MAE parcial=309.372


C:\Users\brunacardoso-ieg\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\brunacardoso-ieg\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\brunacardoso-ieg\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\U

SARIMAX: 120/258 origens | MAE parcial=316.152


...

C:\Users\brunacardoso-ieg\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


SARIMAX: 200/258 origens | MAE parcial=291.367
SARIMAX: 220/258 origens | MAE parcial=296.473
SARIMAX: 240/258 origens | MAE parcial=292.741


C:\Users\brunacardoso-ieg\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


SARIMAX: 258/258 origens | MAE parcial=292.478
Vencedor: {'order': [2, 0, 1], 'seasonal_order': [0, 1, 1, 24]}


,model,parameters,validation_mae,mean_seconds_per_origin,elapsed_seconds,convergence_rate,status,protocol_signature,order_complexity,max_order,mae_3
0,SARIMAX,"{""order"": [2, 0, 1], ""seasonal_order"": [0, 1, ...",292.478034,37.866843,9769.645429,0.972868,ok,e071c45707f5eb8d,5,2,292.478
1,SARIMAX,"{""order"": [2, 0, 0], ""seasonal_order"": [0, 1, ...",293.487683,35.202483,9082.240628,1.000000,ok,e071c45707f5eb8d,4,2,293.488
2,SARIMAX,"{""order"": [2, 1, 2], ""seasonal_order"": [0, 1, ...",295.181683,38.003192,9804.823444,1.000000,ok,e071c45707f5eb8d,7,2,295.182
3,SARIMAX,"{""order"": [2, 1, 1], ""seasonal_order"": [0, 1, ...",296.055105,71.715859,18502.691662,1.000000,ok,e071c45707f5eb8d,6,2,296.055
4,SARIMAX,"{""order"": [2, 0, 2], ""seasonal_order"": [0, 1, ...",298.226650,44.373458,11448.352195,0.918605,ok,e071c45707f5eb8d,6,2,298.227


O teste final não foi usado nesta escolha. Revise o ranking antes de executar o notebook de walk-forward final.